In [ ]:
!pip install torch_geometric ogb

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv

class VanillaGIN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.5):
        super().__init__()

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for i in range(num_layers):
            layer_in = in_channels if i == 0 else hidden_channels
            layer_out = hidden_channels 

            mlp = nn.Sequential(
                nn.Linear(layer_in, hidden_channels),
                nn.BatchNorm1d(hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, layer_out)
            )
            
            self.convs.append(GINConv(mlp, train_eps=True))
            
            if i < num_layers - 1:
                self.bns.append(nn.BatchNorm1d(hidden_channels))

        self.post_lp = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.convs[-1](x, edge_index)
        return x

In [ ]:
from torch_geometric.utils import negative_sampling
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator
import time

_original_load = torch.load
def _patched_load(f, **kwargs):
    kwargs["weights_only"] = False
    return _original_load(f, **kwargs)

torch.load = _patched_load

dataset = PygLinkPropPredDataset(name="ogbl-collab")
split_edge = dataset.get_edge_split()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

train_edges    = split_edges['train']['edge'].to(device)
val_edges_pos  = split_edges['valid']['edge'].to(device)
val_edges_neg  = split_edges['valid']['edge_neg'].to(device)
test_edges_pos = split_edges['test']['edge'].to(device)
test_edges_neg = split_edges['test']['edge_neg'].to(device)

print(f"val_edges_neg shape:  {val_edges_neg.shape}")
print(f"test_edges_neg shape: {test_edges_neg.shape}")
print(f"Train edges: {train_edges.shape}")
print(f"Val edges: {val_edges_pos.shape}")
print(f"Test edges: {test_edges_pos.shape}")

def score_edges(z, edges):
    return (z[edges[:, 0]] * z[edges[:, 1]]).sum(dim=-1)

In [ ]:
evaluator = Evaluator(name='ogbl-collab')

def train(model, optimizer, criterion):
    model.train()
    optimizer.zero_grad()

    z = model(data.x, data.edge_index)

    pos_score = score_edges(z, train_edges)

    neg_edge_index = negative_sampling(
        edge_index=data.edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=train_edges.size(0)
    )
    neg_score = score_edges(z, neg_edge_index.t())

    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([
        torch.ones(pos_score.size(0)),
        torch.zeros(neg_score.size(0))
    ]).to(device)

    loss = criterion(scores, labels)
    loss.backward()
    optimizer.step()

    return loss.item()

@torch.no_grad()
def evaluate(model):
    model.eval()
    z = model(data.x, data.edge_index)

    results = {}
    for split, pos, neg in [
        ('valid', val_edges_pos, val_edges_neg),
        ('test',  test_edges_pos, test_edges_neg)
    ]:
        pos_score = score_edges(z, pos)
        neg_score = score_edges(z, neg)

        hits = evaluator.eval({
            'y_pred_pos': pos_score,
            'y_pred_neg': neg_score,
        })
        results[split] = hits['hits@50']

    return results

In [ ]:
model = VanillaGIN(
    in_channels=dataset.num_features,
    hidden_channels=512,
    num_layers=4,
    dropout=0.5
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

print(model)
print(f"Num parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
NUM_EPOCHS = 200
best_val  = 0
best_test = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t = time.time()
    loss = train(model, optimizer, criterion)
    elapsed = time.time() - t
    scheduler.step()

    if epoch % 10 == 0:
        results = evaluate(model)

        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
              f"Val Hits@50: {results['valid']:.4f} | "
              f"Test Hits@50: {results['test']:.4f} | "
              f"Time: {elapsed:.2f}s")

        if results['valid'] > best_val:
            best_val  = results['valid']
            best_test = results['test']
            torch.save(model.state_dict(), 'best_gin_vanilla_link.pt')

print(f"\nBest Val  Hits@50: {best_val:.4f}")
print(f"Best Test Hits@50: {best_test:.4f}")